# Adult Census Income – Full ML Pipeline
Predicts whether a person earns **>50K** or **≤50K** per year using the UCI Adult Census Income dataset.

| Column | Description | Type |
|---|---|---|
| age | Age of the person | Numeric |
| workclass | Employer type (Private, Gov, …) | Categorical |
| fnlwgt | Census statistical weight | Numeric |
| education | Education level | Categorical |
| education.num | Numeric encoding of education | Numeric |
| marital.status | Marital status | Categorical |
| occupation | Job type | Categorical |
| relationship | Family role | Categorical |
| race | Race | Categorical |
| sex | Sex | Binary |
| capital.gain | Capital gains | Numeric |
| capital.loss | Capital losses | Numeric |
| hours.per.week | Weekly working hours | Numeric |
| native.country | Country of origin | Categorical |
| income | Target: income level | Binary |


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Load Data
> Update the path below to point to your CSV file.

In [ ]:
# Update this path if running locally
# df = pd.read_csv('Adult_Census_Income.csv', na_values=['?'])
# For Google Colab:
# df = pd.read_csv('/content/Adult Census Income.csv', na_values=['?'])

df = pd.read_csv('Adult_Census_Income.csv', na_values=['?'])
print('Shape:', df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
df.isnull().sum().sort_values(ascending=False)

In [ ]:
print('Duplicates:', df.duplicated().sum())

In [ ]:
# Target distribution
sns.countplot(x='income', data=df, palette='Blues')
plt.title('Income Distribution')
plt.show()

In [ ]:
# Age distribution by income
sns.histplot(data=df, x='age', hue='income', bins=30, palette='Blues')
plt.title('Age Distribution by Income')
plt.show()

In [ ]:
# Hours per week vs age, coloured by income
sns.scatterplot(data=df, x='hours.per.week', y='age', hue='income', alpha=0.4, palette='Blues')
plt.title('Hours/Week vs Age by Income')
plt.show()

In [ ]:
# Sex count by income
sns.countplot(data=df, x='sex', hue='income', palette='Blues')
plt.title('Sex vs Income')
plt.show()

In [ ]:
# Race count by income
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='race', hue='income', palette='Blues')
plt.xticks(rotation=45)
plt.title('Race vs Income')
plt.show()

In [ ]:
# Capital gain scatter
sns.scatterplot(data=df, y='capital.gain', x='income', hue='education', alpha=0.5)
plt.title('Capital Gain by Income & Education')
plt.show()

## 4. Preprocessing

In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print('Shape after dedup:', df.shape)

In [ ]:
# Split features and target
X = df.drop('income', axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=15, stratify=y)

for frame in [X_train, X_test, y_train, y_test]:
    frame.reset_index(drop=True, inplace=True)

print('Train size:', X_train.shape, '| Test size:', X_test.shape)

In [ ]:
# Impute missing values in categorical columns
impute_cols = ['occupation', 'workclass', 'native.country']
imputer = SimpleImputer(strategy='most_frequent')
X_train[impute_cols] = imputer.fit_transform(X_train[impute_cols])
X_test[impute_cols]  = imputer.transform(X_test[impute_cols])

print('Remaining NaNs – train:', X_train.isnull().sum().sum(),
      '| test:', X_test.isnull().sum().sum())

In [ ]:
# Drop high-cardinality columns (education already encoded numerically; native.country too sparse)
drop_cols = ['education', 'native.country']
X_train.drop(drop_cols, axis=1, inplace=True)
X_test.drop(drop_cols,  axis=1, inplace=True)
print('Remaining columns:', X_train.columns.tolist())

In [ ]:
# Encode categorical columns – one LabelEncoder per column (FIX vs original notebook)
cat_cols = X_train.select_dtypes(include=['object', 'str']).columns.tolist()
print('Categorical columns to encode:', cat_cols)

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    # Safe transform: unseen labels mapped to -1 instead of raising ValueError
    X_test[col] = X_test[col].map(
        lambda x, _le=le: int(_le.transform([x])[0]) if x in _le.classes_ else -1
    )
    label_encoders[col] = le

In [ ]:
# Encode target
le_y = LabelEncoder()
y_train_enc = le_y.fit_transform(y_train)
y_test_enc  = le_y.transform(y_test)
print('Classes:', le_y.classes_)

In [ ]:
# Correlation heatmap (after encoding)
corr = np.corrcoef(X_train, rowvar=False)
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=X_train.columns, yticklabels=X_train.columns)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

## 5. Model Training & Evaluation

### 5.1 Logistic Regression

In [ ]:
log_reg = LogisticRegression(random_state=15, solver='liblinear', max_iter=1000)
log_reg.fit(X_train_s, y_train_enc)
y_pred_lr = log_reg.predict(X_test_s)

print('Logistic Regression Accuracy:', accuracy_score(y_test_enc, y_pred_lr))
print(classification_report(y_test_enc, y_pred_lr, target_names=le_y.classes_))

sns.heatmap(confusion_matrix(y_test_enc, y_pred_lr), annot=True, fmt='d',
            cmap='Blues', cbar=False, xticklabels=le_y.classes_, yticklabels=le_y.classes_)
plt.title('Logistic Regression – Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

### 5.2 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=15)
dt.fit(X_train_s, y_train_enc)
y_pred_dt = dt.predict(X_test_s)

print('Decision Tree Accuracy:', accuracy_score(y_test_enc, y_pred_dt))
print(classification_report(y_test_enc, y_pred_dt, target_names=le_y.classes_))

sns.heatmap(confusion_matrix(y_test_enc, y_pred_dt), annot=True, fmt='d',
            cmap='Blues', cbar=False, xticklabels=le_y.classes_, yticklabels=le_y.classes_)
plt.title('Decision Tree – Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

### 5.3 Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=15, n_estimators=100)
rf.fit(X_train_s, y_train_enc)
y_pred_rf = rf.predict(X_test_s)

print('Random Forest Accuracy:', accuracy_score(y_test_enc, y_pred_rf))
print(classification_report(y_test_enc, y_pred_rf, target_names=le_y.classes_))

sns.heatmap(confusion_matrix(y_test_enc, y_pred_rf), annot=True, fmt='d',
            cmap='Blues', cbar=False, xticklabels=le_y.classes_, yticklabels=le_y.classes_)
plt.title('Random Forest – Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

### 5.4 K-Nearest Neighbors

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train_enc)
y_pred_knn = knn.predict(X_test_s)

print('KNN Accuracy:', accuracy_score(y_test_enc, y_pred_knn))
print(classification_report(y_test_enc, y_pred_knn, target_names=le_y.classes_))

sns.heatmap(confusion_matrix(y_test_enc, y_pred_knn), annot=True, fmt='d',
            cmap='Blues', cbar=False, xticklabels=le_y.classes_, yticklabels=le_y.classes_)
plt.title('KNN – Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

## 6. Model Comparison

In [ ]:
results = {
    'Logistic Regression': accuracy_score(y_test_enc, y_pred_lr),
    'Decision Tree':       accuracy_score(y_test_enc, y_pred_dt),
    'Random Forest':       accuracy_score(y_test_enc, y_pred_rf),
    'KNN':                 accuracy_score(y_test_enc, y_pred_knn),
}

results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
results_df.sort_values('Accuracy', ascending=False, inplace=True)
print(results_df.to_string(index=False))

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, x='Model', y='Accuracy', palette='Blues_d')
plt.ylim(0.75, 0.90)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=15)
for i, row in results_df.reset_index().iterrows():
    plt.text(i, row['Accuracy'] + 0.001, f"{row['Accuracy']:.4f}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Save Best Model (for Deployment)

In [ ]:
import joblib, os

os.makedirs('artifacts', exist_ok=True)

# Save best model (Random Forest) and all pipeline components
joblib.dump(rf,             'artifacts/model.pkl')
joblib.dump(scaler,         'artifacts/scaler.pkl')
joblib.dump(imputer,        'artifacts/imputer.pkl')
joblib.dump(label_encoders, 'artifacts/label_encoders.pkl')
joblib.dump(le_y,           'artifacts/label_encoder_y.pkl')
joblib.dump(X_train.columns.tolist(), 'artifacts/feature_cols.pkl')

print('Saved artefacts:', os.listdir('artifacts'))